In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib, os
os.makedirs('./models', exist_ok=True)

# 데이터 로드
train  = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/train.csv')
test   = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/test.csv')
layout = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/layout_info.csv')

# layout merge
layout_clean = layout[['layout_id', 'layout_type', 'aisle_width_avg',
                        'intersection_count', 'one_way_ratio', 'pack_station_count',
                        'charger_count', 'layout_compactness', 'zone_dispersion',
                        'robot_total', 'floor_area_sqm', 'ceiling_height_m',
                        'fire_sprinkler_count', 'emergency_exit_count']]

train = train.merge(layout_clean, on='layout_id', how='left')
test  = test.merge(layout_clean, on='layout_id', how='left')

le = LabelEncoder()
train['layout_type'] = le.fit_transform(train['layout_type'].fillna('unknown'))
test['layout_type']  = le.transform(test['layout_type'].fillna('unknown'))

TARGET   = 'avg_delay_minutes_next_30m'
ID_COLS  = ['ID', 'layout_id', 'scenario_id']
feature_cols = [c for c in train.columns if c not in ID_COLS + [TARGET]]

from xgboost import XGBRegressor

X = train[feature_cols]
y = train[TARGET]

import cupy as cp

# test 데이터를 GPU로 올리기
test_gpu = cp.array(test[feature_cols].values)

xgb_preds = np.zeros(len(test))
for fold in range(1, 6):
    m = joblib.load(f'C:/Users/user/dacon/Smart_Logistics/models/xgb_fold{fold}.pkl')
    xgb_preds += m.predict(test_gpu) / 5

# LightGBM 저장된 모델로 예측
lgbm_preds = np.zeros(len(test))
for fold in range(1, 6):
    m = joblib.load(f'C:/Users/user/dacon/Smart_Logistics/models/lgbm_simple_fold{fold}.pkl')
    lgbm_preds += m.predict(test[feature_cols]) / 5

# 앙상블
ensemble_preds = (lgbm_preds + xgb_preds) / 2

test_id = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/test.csv')['ID']
submission = pd.DataFrame({
    'ID'                        : test_id,
    'avg_delay_minutes_next_30m': ensemble_preds
})

submission.to_csv('C:/Users/user/dacon/Smart_Logistics/results/submission_v13_lgbm_xgb_ensemble.csv', index=False)
print(submission.head())

c:\Users\user\anaconda3\envs\a\lib\site-packages\xgboost\core.py:751: UserWarning: [14:57:31] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


            ID  avg_delay_minutes_next_30m
0  TEST_000000                   15.567971
1  TEST_000001                   15.080583
2  TEST_000002                   18.596137
3  TEST_000003                   18.053804
4  TEST_000004                   17.004647
